# Getting securities lending trade by trade data

**What ?** Securities lending trade-by-trade refers to contracts in which securities, such as stocks or bonds, are lent by one party (the lender/donor) to another (the borrower/taker) in exchange for collateral. These contracts indicate that the borrower has temporary possession of the securities and must return them to the lender at the end of the specified period. The Brazilian stock exchange B3 provides daily trade-by-trade data on all lending contracts traded over-the-counter (OTC) and via the electronic trading screen. This data includes the traded hour, tickers, quantities, rates, and the brokers on both the lender and borrower sides. [[see glossary here]](https://www.b3.com.br/data/files/A6/C3/43/9D/3941B810E9C1AAA8AC094EA8/Negocio%20a%20Negocio%20-%20Emprestimos%20de%20Ativos%20-%20btb%202023.pdf).

**Why ?** Securities lending registers data is crucial for financial market analysis related on market sentiment indicator, supply and demand insights, risk management and price movement predictions.

**How ?** The leading trade-by-trade data are available on the B3 website as a daily .csv file [link here](https://www.b3.com.br/pt_br/market-data-e-indices/servicos-de-dados/market-data/consultas/boletim-diario/boletim-diario-do-mercado/). This file was manually downloaded to a local directory. The provided Python code is responsible for reading this data, applying necessary data cleaning and transformation, and then saving the processed dataframe into a local SQLite database for further analysis.

<img src="https://lh3.googleusercontent.com/d/1iTe-yT-8lTy6z5aUDMM5EAuU663wHvHk" alt="texto_alternativo" width="200" align="center">


## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import re

import sqlite3
import requests
import zipfile
from datetime import datetime

#### Search at local SQLite database what is the last available data uploaded 

In [7]:
conn = sqlite3.connect(os.getenv('MY_FINANCE_DB_PATH')+'/finance_database.db')
cursor = conn.cursor()
cursor.execute('''SELECT ClosedDate 
                    FROM B3_securities_lending_trade_by_trade 
                    GROUP BY ClosedDate ''') # this table was previously created to hold the trade by trade data

rows = cursor.fetchall()
columns = [description[0] for description in cursor.description]

df_dt = pd.DataFrame(rows, columns=columns)
conn.close()

df_dt['ClosedDate'].sort_values(ascending = False).head(3)

32    2024-08-06
31    2024-08-05
30    2024-08-02
Name: ClosedDate, dtype: object

####  Looking for new files manually downloaded from B3 website into a local folder

In [2]:
file_path = r'C:\Users\lucas\OneDrive\CM_Explorer\data_scraping\B3_Daily_market_bulletin\temp_file\Clearing\Emprestimo_de_Ativos_Neg_a_Neg'
all_files = [file for file in os.listdir(file_path) if "Empréstimos de Ativos – Negócios" in file]
all_files

['Empréstimos de Ativos – Negócios-01-07-2024.csv',
 'Empréstimos de Ativos – Negócios-01-08-2024.csv',
 'Empréstimos de Ativos – Negócios-02-07-2024.csv',
 'Empréstimos de Ativos – Negócios-02-08-2024.csv',
 'Empréstimos de Ativos – Negócios-03-07-2024.csv',
 'Empréstimos de Ativos – Negócios-04-07-2024.csv',
 'Empréstimos de Ativos – Negócios-05-07-2024.csv',
 'Empréstimos de Ativos – Negócios-05-08-2024.csv',
 'Empréstimos de Ativos – Negócios-06-08-2024.csv',
 'Empréstimos de Ativos – Negócios-08-07-2024.csv',
 'Empréstimos de Ativos – Negócios-09-07-2024.csv',
 'Empréstimos de Ativos – Negócios-10-07-2024.csv',
 'Empréstimos de Ativos – Negócios-11-07-2024.csv',
 'Empréstimos de Ativos – Negócios-12-07-2024.csv',
 'Empréstimos de Ativos – Negócios-15-07-2024.csv',
 'Empréstimos de Ativos – Negócios-16-07-2024.csv',
 'Empréstimos de Ativos – Negócios-17-07-2024.csv',
 'Empréstimos de Ativos – Negócios-18-07-2024.csv',
 'Empréstimos de Ativos – Negócios-19-07-2024.csv',
 'Empréstimo

#### Extract, Transform and Load dataset

In [3]:
#defining new columns names
new_col_names = {'Papel':'ticker',
             'Quantidade':'asset_qty',
             'Taxa % Remuneração':'trade_rate',
             'Número do negócio':'trade_number_id',
             'Mercado':'market',
             'Data de referência':'ClosedDate',
             'Hora':'ClosedHour',
             'Ação de Atualização':'update_action',
             'Tipo Sessão do Pregão':'session_type',
             'Código':'broker_donor_id',
             'Nome Doador':'broker_donor_name',
             'Código.1':'broker_taker_id',
             'Nome Tomador':'broker_taker_name'}

# Reading each .csv manually downloaded from B3
####################################################################################################
df_app = pd.DataFrame()

for files in all_files: #zip_files_with_paths[::-1][6:8]

    df_app = pd.read_csv(file_path+'/'+files,sep = ";", encoding = "UTF-8", skiprows = 1, low_memory=False, dtype=str)
    
# Data cleaning: changing columns names and data types
####################################################################################################

    # rename columns    
    df_app.rename(columns = new_col_names, inplace = True)
    
    # changing datatypes
    df_app['asset_qty'] = df_app['asset_qty'].str.replace('.', '',regex=False).astype(float)
    df_app['trade_rate'] = df_app['trade_rate'].str.replace('%', '',regex=False).str.replace(',', '.',regex=False).astype(float)

    df_app['ClosedDate'] = pd.to_datetime(df_app['ClosedDate'], format='%d/%m/%Y')
    df_app['ClosedDate'] = df_app['ClosedDate'].dt.strftime('%Y-%m-%d')

    # adding a processing data column to referece the date of the scraping process run
    df_app['proc_datedt'] = datetime.now().replace(microsecond=0) 

# Write the dataframe into the SQLite database
####################################################################################################
    #open connection with local SQLite database
    conn = sqlite3.connect(os.getenv('MY_FINANCE_DB_PATH')+'/finance_database.db')
    df_app.to_sql('B3_securities_lending_trade_by_trade', conn, if_exists='append',index=False)
    
    # printing files read over each iteraction
    print(files)
    del df_app 
    df_app = pd.DataFrame()
    
conn.close()

Empréstimos de Ativos – Negócios-01-07-2024.csv
Empréstimos de Ativos – Negócios-01-08-2024.csv
Empréstimos de Ativos – Negócios-02-07-2024.csv
Empréstimos de Ativos – Negócios-02-08-2024.csv
Empréstimos de Ativos – Negócios-03-07-2024.csv
Empréstimos de Ativos – Negócios-04-07-2024.csv
Empréstimos de Ativos – Negócios-05-07-2024.csv
Empréstimos de Ativos – Negócios-05-08-2024.csv
Empréstimos de Ativos – Negócios-06-08-2024.csv
Empréstimos de Ativos – Negócios-08-07-2024.csv
Empréstimos de Ativos – Negócios-09-07-2024.csv
Empréstimos de Ativos – Negócios-10-07-2024.csv
Empréstimos de Ativos – Negócios-11-07-2024.csv
Empréstimos de Ativos – Negócios-12-07-2024.csv
Empréstimos de Ativos – Negócios-15-07-2024.csv
Empréstimos de Ativos – Negócios-16-07-2024.csv
Empréstimos de Ativos – Negócios-17-07-2024.csv
Empréstimos de Ativos – Negócios-18-07-2024.csv
Empréstimos de Ativos – Negócios-19-07-2024.csv
Empréstimos de Ativos – Negócios-21-06-2024.csv
Empréstimos de Ativos – Negócios-22-07-2

#### Reading a sample from the SQLite database to check the upload

In [5]:
# Show a sample of the data
conn = sqlite3.connect(os.getenv('MY_FINANCE_DB_PATH')+'/finance_database.db')
cursor = conn.cursor()
cursor.execute('''SELECT *
            FROM B3_securities_lending_trade_by_trade
            WHERE ClosedDate = '2024-07-01' ''') # reading a specifically ticker and date as an exlaple
rows = cursor.fetchall()
columns = [description[0] for description in cursor.description]

df_sample = pd.DataFrame(rows, columns=columns)
conn.close()

df_sample.head()

,ticker,asset_qty,trade_rate,trade_number_id,market,ClosedDate,ClosedHour,update_action,session_type,broker_donor_id,broker_donor_name,broker_taker_id,broker_taker_name,proc_datedt
0,NTCO3,92015.0,0.11,62.198.187,Balcão,2024-07-01,19:28:27,Novo (0),Regular (1),59,SAFRA CORRETORA DE VALORES E CAMBIO LTDA,40,MORGAN STANLEY CTVM S/A,2024-08-13 07:06:17
1,SLCE3,12135.0,0.16,62.197.389,Balcão,2024-07-01,19:28:27,Novo (0),Regular (1),27,SANTANDER CCVM S/A,40,MORGAN STANLEY CTVM S/A,2024-08-13 07:06:17
2,AERI3,1719.0,1.57,62.197.812,Balcão,2024-07-01,19:28:27,Novo (0),Regular (1),4090,TORO CTVM SA,40,MORGAN STANLEY CTVM S/A,2024-08-13 07:06:17
3,SLCE3,2228.0,0.16,62.197.381,Balcão,2024-07-01,19:28:27,Novo (0),Regular (1),27,SANTANDER CCVM S/A,40,MORGAN STANLEY CTVM S/A,2024-08-13 07:06:17
4,SLCE3,273.0,0.16,62.197.065,Balcão,2024-07-01,19:28:27,Novo (0),Regular (1),59,SAFRA CORRETORA DE VALORES E CAMBIO LTDA,40,MORGAN STANLEY CTVM S/A,2024-08-13 07:06:17
